# Depth-Anything-V2 Large với mask DO MODEL DỰ ĐOÁN (food_seg_v4_final.pt, conf 0.25)

Với mỗi ảnh có cân (112 ảnh):
1. Depth-Anything-V2 (Small và Large) -> bản đồ disparity tương đối (lớn = gần camera).
2. Fit mặt phẳng lên **mặt đĩa lộ ra** (mask YOLOE, dataset `food-plate-masks`) -> độ cao so với đĩa `h = d - mặt_phẳng`.
3. Với mỗi món (mask nhãn tay, hợp theo lớp): tổng / trung bình / p90 của `h`, theo 3 cách chuẩn hoá thang đo.

Notebook chỉ xuất đặc trưng (`depth_features.csv`). Phần phân tích (CV theo nhóm, so với diện tích) làm ở máy, giống `grams.py`.

In [ ]:
!pip install -q ultralytics
import glob, os, pathlib, yaml, numpy as np, pandas as pd, cv2, torch
from ultralytics import YOLO
SEG = YOLO(glob.glob('/kaggle/input/**/food_seg_v4_final.pt', recursive=True)[0])
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
SRC = str(pathlib.Path([p for p in glob.glob('/kaggle/input/**/data.yaml', recursive=True)
                        if 'test' in yaml.safe_load(open(p))][0]).parent)
PM = str(pathlib.Path(glob.glob('/kaggle/input/**/plate_masks', recursive=True)[0]))
cfg = yaml.safe_load(open(f'{SRC}/data.yaml')); NAMES = [cfg['names'][i] for i in sorted(cfg['names'])]
assert len(NAMES) == 45, len(NAMES)
man = pd.read_csv(f'{SRC}/manifest.csv'); split = dict(zip(man.image, man.split))
imgs = sorted(set(pd.read_csv(f'{SRC}/weights.csv').image))
print(SRC, PM, len(imgs), 'ảnh có cân', torch.cuda.is_available())

In [ ]:
def fit_plane(d, ring, iters=4):
    ys, xs = np.nonzero(ring)
    if len(xs) < 200: return None
    if len(xs) > 40000:
        sel = np.random.default_rng(0).choice(len(xs), 40000, replace=False); ys, xs = ys[sel], xs[sel]
    z = d[ys, xs].astype(np.float64); A = np.c_[xs, ys, np.ones_like(xs)].astype(np.float64)
    keep = np.ones(len(z), bool)
    for _ in range(iters):
        coef, *_ = np.linalg.lstsq(A[keep], z[keep], rcond=None)
        r = z - A @ coef; mad = np.median(np.abs(r[keep] - np.median(r[keep]))) + 1e-9
        keep = np.abs(r) < 3 * 1.4826 * mad
    H, W = d.shape; yy, xx = np.mgrid[0:H, 0:W]
    return coef[0] * xx + coef[1] * yy + coef[2], z[keep]

def class_masks(im, H, W):
    out = {}
    for l in open(f'{SRC}/labels/{split[im]}/{im[:-4]}.txt'):
        r = l.split()
        if not r: continue
        xy = (np.array(r[1:], float).reshape(-1, 2) * [W, H]).round().astype(np.int32)
        cv2.fillPoly(out.setdefault(NAMES[int(r[0])], np.zeros((H, W), np.uint8)), [xy], 1)
    return {k: v > 0 for k, v in out.items()}

MODELS = {'L': 'depth-anything/Depth-Anything-V2-Large-hf'}
def pred_masks(p, H, W):
    r = SEG.predict(p, conf=0.25, retina_masks=True, verbose=False)[0]; out = {}
    if r.masks is not None:
        for m, c in zip(r.masks.data.cpu().numpy() > 0.5, r.boxes.cls.cpu().numpy().astype(int)):
            n = SEG.names[c]; out[n] = out.get(n, np.zeros((H, W), bool)) | m
    return out
rows = []; os.makedirs('/kaggle/working/vis', exist_ok=True)
for tag, name in MODELS.items():
    proc = AutoImageProcessor.from_pretrained(name); model = AutoModelForDepthEstimation.from_pretrained(name).cuda().eval()
    for i, im in enumerate(imgs):
        bgr = cv2.imread(f'{SRC}/images/{split[im]}/{im}'); H, W = bgr.shape[:2]
        with torch.no_grad():
            inp = proc(images=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), return_tensors='pt').to('cuda')
            pd_ = model(**inp).predicted_depth
            d = torch.nn.functional.interpolate(pd_.unsqueeze(1), size=(H, W), mode='bicubic', align_corners=False)[0, 0].float().cpu().numpy()
        pmp = f'{PM}/{im[:-4]}.png'
        ring = cv2.imread(pmp, 0) > 0 if os.path.exists(pmp) else np.zeros((H, W), bool)
        ring = cv2.erode(ring.astype(np.uint8), np.ones((9, 9), np.uint8)) > 0
        fp = fit_plane(d, ring)
        if fp is None:
            rows.append(dict(model=tag, image=im, plane_ok=False)); continue
        plane, ringvals = fp
        h = d - plane
        # thang đo: (a) thô, (b) chia disparity trung vị của đĩa, (c) chia độ trải disparity trong vùng đĩa
        hull = np.zeros((H, W), np.uint8)
        cs = cv2.findContours((cv2.imread(pmp, 0) > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        cv2.fillPoly(hull, [cv2.convexHull(np.vstack(cs))], 1); hull = hull > 0
        norms = {'raw': 1.0, 'dmed': float(np.median(ringvals)), 'spread': float(np.percentile(d[hull], 95) - np.percentile(d[hull], 5)) + 1e-6}
        ring_noise = float(np.std(h[ring]))
        for c, m in pred_masks(f'{SRC}/images/{split[im]}/{im}', H, W).items():
            hv = h[m]; r = dict(model=tag, image=im, **{'class': c}, plane_ok=True, area_px=int(m.sum()), ring_noise=ring_noise)
            for k, s in norms.items():
                x = hv / s
                r[f'hsum_{k}'] = float(np.clip(x, 0, None).sum()); r[f'hmean_{k}'] = float(x.mean()); r[f'hp90_{k}'] = float(np.percentile(x, 90))
            rows.append(r)
        if tag == 'L' and i < 6:
            v = np.clip(h / norms['spread'], -0.3, 1.0); v = ((v + 0.3) / 1.3 * 255).astype(np.uint8)
            cv2.imwrite(f'/kaggle/working/vis/{im[:-4]}_h.jpg', np.hstack([cv2.resize(bgr, (W // 2, H // 2)), cv2.resize(cv2.applyColorMap(v, cv2.COLORMAP_INFERNO), (W // 2, H // 2))]))
    del model; torch.cuda.empty_cache()
    pd.DataFrame(rows).to_csv('/kaggle/working/depth_features_pred.csv', index=False)
    print(tag, 'xong', len(rows))

In [ ]:
F = pd.read_csv('/kaggle/working/depth_features_pred.csv')
print(F.groupby('model').plane_ok.mean())
print(F[F['class'] == 'rice'].groupby('model')[['hmean_spread', 'hp90_spread', 'ring_noise']].describe().T.round(3))